In [39]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Lasso

In [40]:
from google.colab import files

uploaded = files.upload()

Saving BT2024129_train_var1.csv to BT2024129_train_var1 (2).csv


In [41]:
df = pd.read_csv("BT2024129_train_var1.csv")


In [42]:
X = df.drop(columns=["y"])
y = df["y"]

In [43]:
print(df.head())
print(df.shape)
print(df.columns)

         x1        x2        x3        x4        x5        x6         y
0  0.408928  0.158219  0.125688 -1.000000  0.426643  0.338822 -0.629294
1  0.382004 -0.488516 -1.000000 -0.888032 -1.000000 -1.000000  4.601971
2  0.316776  0.118727 -0.707126 -0.027580  0.891690 -0.661984 -1.277367
3  1.000000  1.000000  0.242794  1.000000  1.000000  0.733250 -2.488085
4  0.048394  0.112756  0.527160 -0.840413  0.732319 -0.531173  2.212094
(1000, 7)
Index(['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'y'], dtype='object')


In [44]:
print(X.shape)
print(y.shape)

(1000, 6)
(1000,)


In [45]:
pipeline = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

In [46]:
param_grid = {
    "poly__degree": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "ridge__alpha": [0, 1e-6, 1e-5, 1e-4, 1e-3,
                     1e-2, 1e-1, 1, 10, 100]
}

In [47]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [48]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=kf,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

In [49]:
grid_search.fit(X, y)

GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('scaler', StandardScaler()),
                                       ('ridge', Ridge())]),
             n_jobs=-1,
             param_grid={'poly__degree': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                         'ridge__alpha': [0, 1e-06, 1e-05, 0.0001, 0.001, 0.01,
                                          0.1, 1, 10, 100]},
             scoring='neg_mean_squared_error')

In [50]:
print("Best parameters:")
print(grid_search.best_params_)

Best parameters:
{'poly__degree': 5, 'ridge__alpha': 10}


In [51]:
best_cv_mse = -grid_search.best_score_
print("Best CV MSE:", best_cv_mse)

Best CV MSE: 0.5348429884227253


In [52]:
best_model = grid_search.best_estimator_

In [53]:
#predictions on training ds
y_pred_train = best_model.predict(X)
train_mse = mean_squared_error(y, y_pred_train)
print("Training MSE:", train_mse)

Training MSE: 0.18221823361276285


In [54]:
fine_alphas = np.linspace(5, 15, 21)
fine_param_grid = {
    "poly__degree": [5],
    "ridge__alpha": fine_alphas
}

fine_grid = GridSearchCV(
    estimator=pipeline,
    param_grid=fine_param_grid,
    cv=kf,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

In [55]:
fine_grid.fit(X, y)

GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('scaler', StandardScaler()),
                                       ('ridge', Ridge())]),
             n_jobs=-1,
             param_grid={'poly__degree': [5],
                         'ridge__alpha': array([ 5. ,  5.5,  6. ,  6.5,  7. ,  7.5,  8. ,  8.5,  9. ,  9.5, 10. ,
       10.5, 11. , 11.5, 12. , 12.5, 13. , 13.5, 14. , 14.5, 15. ])},
             scoring='neg_mean_squared_error')

In [56]:
print("Best parameters:", fine_grid.best_params_)
print("Best CV MSE:", -fine_grid.best_score_)

Best parameters: {'poly__degree': 5, 'ridge__alpha': np.float64(15.0)}
Best CV MSE: 0.5205176502585279


In [57]:
fine_alphas_2 = np.linspace(10, 30, 41)

fine_param_grid_2 = {
    "poly__degree": [5],
    "ridge__alpha": fine_alphas_2
}

fine_grid_2 = GridSearchCV(
    pipeline,
    fine_param_grid_2,
    cv=kf,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

fine_grid_2.fit(X, y)

print("Best parameters:", fine_grid_2.best_params_)
print("Best CV MSE:", -fine_grid_2.best_score_)

Best parameters: {'poly__degree': 5, 'ridge__alpha': np.float64(21.0)}
Best CV MSE: 0.5167760973379784


In [58]:
param_grid_degree = {
    "poly__degree": range(1, 11),
    "ridge__alpha": [15.5]
}

degree_search = GridSearchCV(
    pipeline,
    param_grid_degree,
    cv=kf,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

degree_search.fit(X, y)

print(degree_search.best_params_)
print(-degree_search.best_score_)

{'poly__degree': 5, 'ridge__alpha': 15.5}
0.5198400135499606


In [59]:
best_model = degree_search.best_estimator_

In [60]:
y_pred_train = best_model.predict(X)

train_mse = mean_squared_error(y, y_pred_train)

print("Training MSE:", train_mse)
print("CV MSE:", -degree_search.best_score_)

Training MSE: 0.19357532237605063
CV MSE: 0.5198400135499606


In [61]:
lasso_pipeline = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=2000))
])

lasso_grid = {
    "poly__degree": range(1, 7),
    "lasso__alpha": np.logspace(-4, 1, 15)
}

lasso_search = GridSearchCV(
    lasso_pipeline,
    lasso_grid,
    cv=kf,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

In [62]:
lasso_search.fit(X, y)

GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('scaler', StandardScaler()),
                                       ('lasso', Lasso(max_iter=2000))]),
             n_jobs=-1,
             param_grid={'lasso__alpha': array([1.00000000e-04, 2.27584593e-04, 5.17947468e-04, 1.17876863e-03,
       2.68269580e-03, 6.10540230e-03, 1.38949549e-02, 3.16227766e-02,
       7.19685673e-02, 1.63789371e-01, 3.72759372e-01, 8.48342898e-01,
       1.93069773e+00, 4.39397056e+00, 1.00000000e+01]),
                         'poly__degree': range(1, 7)},
             scoring='neg_mean_squared_error')

In [63]:
print(lasso_search.best_params_)
print("CV MSE:", -lasso_search.best_score_)

{'lasso__alpha': np.float64(0.013894954943731374), 'poly__degree': 5}
CV MSE: 0.3381889241364483


In [64]:
uploaded2 = files.upload()

Saving BT2024129_test_var1.csv to BT2024129_test_var1 (2).csv


In [65]:
test_df = pd.read_csv("BT2024129_test_var1.csv")
X_test=test_df
y_test_pred=lasso_search.best_estimator_.predict(X_test)


In [66]:
output=pd.DataFrame({
    "y":y_test_pred
})

In [67]:
output.to_csv("predictions.csv",index=False)

In [68]:
files.download("predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [69]:
print(lasso_search.best_estimator_.score(X,y))

0.9729499639141561
